In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder ,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostClassifier, Pool 

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# load dataset

In [3]:
df = pd.read_csv("synthetic_health_dataset.csv")
df.head()

,Gender,Smoking,Alcohol_Consumption,Exercise_Frequency,Blood_Pressure,Cholesterol_Level,Stress_Level,Age,BMI,Heart_Rate,Sleep_Hours,Blood_Sugar_Level,Medication_Use,Family_History,Illness
0,Fe__Male__,No,Moderate,Never,0Normal102,Borderline,High,"""90""",16.6,119,3.6,143.6,NaN,No,Yes
1,Other,Yes,NaN,Never,0Normal102,0Normal102,Low,20,29.9,69,9.9,121.8,NaN,Yes,No
2,__Male__,Yes,Heavy,Daily,High,High,Low,52,33.5,54,8.5,107,NaN,Yes,Yes
3,__Male__,Yes,Heavy,Daily,0Normal102,High,Low,15,20.3,72,9.5,92.1,NaN,No,Yes
4,__Male__,No,Moderate,Often,High,High,Medium,60,36.0,58,4.4,113.6,NaN,No,Yes


# DATA CLEANING

In [4]:
df.columns

Index(['Gender', 'Smoking', 'Alcohol_Consumption', 'Exercise_Frequency',
       'Blood_Pressure', 'Cholesterol_Level', 'Stress_Level', 'Age', 'BMI',
       'Heart_Rate', 'Sleep_Hours', 'Blood_Sugar_Level', 'Medication_Use',
       'Family_History', 'Illness'],
      dtype='object')

In [5]:
column_mapping = {
    'Gender': 'Gender',
    'Smoking': 'Smoking',
    'hol_Consump': 'Alcohol_Consumption',
    'rcise_Freque': 'Exercise_Frequency',
    'lood_Pressur': 'Blood_Pressure',
    'olesterol_Lev': 'Cholesterol_Level',
    'Stress_Level': 'Stress_Level',
    'Age': 'Age',
    'BMI': 'BMI',
    'Heart_Rate': 'Heart_Rate',
    'Sleep_Hours': 'Sleep_Hours',
    'od_Sugar_Leve': 'Blood_Sugar_Level',
    'edication_Usa': 'Medication_Usage',
    'amily_Histor': 'Family_History',
    'Illne': 'Illness'
}
df.rename(columns=column_mapping, inplace=True)

In [6]:
df.head()

,Gender,Smoking,Alcohol_Consumption,Exercise_Frequency,Blood_Pressure,Cholesterol_Level,Stress_Level,Age,BMI,Heart_Rate,Sleep_Hours,Blood_Sugar_Level,Medication_Use,Family_History,Illness
0,Fe__Male__,No,Moderate,Never,0Normal102,Borderline,High,"""90""",16.6,119,3.6,143.6,NaN,No,Yes
1,Other,Yes,NaN,Never,0Normal102,0Normal102,Low,20,29.9,69,9.9,121.8,NaN,Yes,No
2,__Male__,Yes,Heavy,Daily,High,High,Low,52,33.5,54,8.5,107,NaN,Yes,Yes
3,__Male__,Yes,Heavy,Daily,0Normal102,High,Low,15,20.3,72,9.5,92.1,NaN,No,Yes
4,__Male__,No,Moderate,Often,High,High,Medium,60,36.0,58,4.4,113.6,NaN,No,Yes


In [7]:
df["Gender"].value_counts()

Gender
__Male__      336
Fe__Male__    326
Other         326
Name: count, dtype: int64

In [8]:
#  Clean string values in 'Gender' 
df['Gender'] = df['Gender'].astype(str).str.replace('__', '', regex=False)
df['Gender'] = df['Gender'].replace({'FeMale': 'Female', 'Male': 'Male', 'Other': 'Other', 'nan': np.nan})

In [15]:
df["Gender"].value_counts()

Gender
Male      336
Female    326
Other     326
Name: count, dtype: int64

In [9]:
df["Blood_Pressure"].value_counts()

Blood_Pressure
0Normal102    342
Low           336
High          312
Name: count, dtype: int64

In [10]:
df["Cholesterol_Level"].value_counts()

Cholesterol_Level
High          342
Borderline    340
0Normal102    311
Name: count, dtype: int64

In [11]:
df["Blood_Pressure"] = df["Blood_Pressure"].replace('0Normal102', 'Normal')
df["Cholesterol_Level"] = df["Cholesterol_Level"].replace('0Normal102', 'Normal')

In [13]:
df["Cholesterol_Level"].value_counts()

Cholesterol_Level
High          342
Borderline    340
Normal        311
Name: count, dtype: int64

#  Clean 'Age' column 

In [15]:
#  Clean 'Age' column 
df['Age'] = df['Age'].astype(str).str.replace('"', '', regex=False).str.replace("'", "", regex=False)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

In [17]:
df["Age"].dtype

dtype('float64')

In [26]:
df['Heart_Rate'] = pd.to_numeric(df['Heart_Rate'], errors='coerce')
df['Blood_Sugar_Level'] = pd.to_numeric(df['Blood_Sugar_Level'], errors='coerce')

In [30]:
df["Medication_Use"].value_counts()

Medication_Use
Occasional       344
Regular          308
Regular____        2
Occasional___      1
Name: count, dtype: int64

In [31]:
df['Medication_Use'] = df['Medication_Use'].astype(str).str.rstrip('_')
df['Medication_Use'] = df['Medication_Use'].replace('nan', np.nan)

#  Fill the missing values with the most common category (Mode)
med_mode = df['Medication_Use'].mode()[0]
df['Medication_Use'] = df['Medication_Use'].fillna(med_mode)

In [32]:
df["Medication_Use"].value_counts()

Medication_Use
Occasional    690
Regular       310
Name: count, dtype: int64

In [33]:
df["Exercise_Frequency"].value_counts()

Exercise_Frequency
Never     258
Daily     248
Often     248
Rarely    239
Name: count, dtype: int64

In [34]:
df.nunique()

Gender                   3
Smoking                  2
Alcohol_Consumption      2
Exercise_Frequency       4
Blood_Pressure           3
Cholesterol_Level        3
Stress_Level             3
Age                    100
BMI                    248
Heart_Rate              70
Sleep_Hours             71
Blood_Sugar_Level      661
Medication_Use           2
Family_History           2
Illness                  2
dtype: int64

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Gender               988 non-null    object 
 1   Smoking              993 non-null    object 
 2   Alcohol_Consumption  649 non-null    object 
 3   Exercise_Frequency   993 non-null    object 
 4   Blood_Pressure       990 non-null    object 
 5   Cholesterol_Level    993 non-null    object 
 6   Stress_Level         995 non-null    object 
 7   Age                  990 non-null    float64
 8   BMI                  992 non-null    float64
 9   Heart_Rate           991 non-null    float64
 10  Sleep_Hours          992 non-null    float64
 11  Blood_Sugar_Level    978 non-null    float64
 12  Medication_Use       655 non-null    object 
 13  Family_History       995 non-null    object 
 14  Illness              1000 non-null   object 
dtypes: float64(5), object(10)
memory usage:

In [37]:
df.duplicated().sum()


np.int64(0)

In [38]:
df.isna().sum()

Gender                  12
Smoking                  7
Alcohol_Consumption    351
Exercise_Frequency       7
Blood_Pressure          10
Cholesterol_Level        7
Stress_Level             5
Age                     10
BMI                      8
Heart_Rate               9
Sleep_Hours              8
Blood_Sugar_Level       22
Medication_Use           0
Family_History           5
Illness                  0
dtype: int64

In [40]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("🤖  Numerical :", numerical_cols)
print("🤖  Categorical :", categorical_cols)

🤖  Numerical : ['Age', 'BMI', 'Heart_Rate', 'Sleep_Hours', 'Blood_Sugar_Level']
🤖  Categorical : ['Gender', 'Smoking', 'Alcohol_Consumption', 'Exercise_Frequency', 'Blood_Pressure', 'Cholesterol_Level', 'Stress_Level', 'Medication_Use', 'Family_History', 'Illness']


In [41]:
# Fill all detected numerical columns with their respective Medians
for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill all detected categorical columns with their respective Modes
for col in categorical_cols:
    # Ensure there is a mode available to prevent crashes on completely empty columns
    if not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode()[0])

In [42]:
df.isnull().sum()

Gender                 0
Smoking                0
Alcohol_Consumption    0
Exercise_Frequency     0
Blood_Pressure         0
Cholesterol_Level      0
Stress_Level           0
Age                    0
BMI                    0
Heart_Rate             0
Sleep_Hours            0
Blood_Sugar_Level      0
Medication_Use         0
Family_History         0
Illness                0
dtype: int64